# Experiment design: power analysis and sample size planning

This notebook covers the pre-experiment planning phase: computing
required sample sizes, selecting the minimum detectable effect (MDE),
estimating test duration, discussing randomization strategy, and
addressing the multiple testing problem.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
sys.path.insert(0, '..')

from src.experiment import power_analysis, multiple_comparison_correction
from src.visualizations import plot_sample_size_vs_mde

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Power analysis

Power analysis determines the minimum sample size needed to detect
a given effect size with a specified probability (power). We use
the standard two-proportion z-test framework.

In [ ]:
# Baseline parameters from our experiments
baseline_rate = 0.03  # 3% baseline conversion
mde = 0.01           # 1 percentage point minimum detectable effect
alpha = 0.05         # 5% significance level
power = 0.80         # 80% power

result = power_analysis(baseline_rate, mde, alpha, power)

print('Power analysis results:')
print(f'  Baseline rate:         {result["baseline_rate"]:.1%}')
print(f'  MDE:                   {result["mde"]:.1%} (absolute)')
print(f'  Relative MDE:          {result["mde"] / result["baseline_rate"]:.1%}')
print(f'  Alpha:                 {result["alpha"]}')
print(f'  Power:                 {result["power"]}')
print(f'  Sample size per group: {result["sample_size_per_group"]:,}')
print(f'  Total sample size:     {result["total_sample_size"]:,}')

## 2. Sample size calculation across MDE values

Smaller effects require larger samples to detect reliably.
This tradeoff is fundamental to experiment planning.

In [ ]:
mde_range = np.arange(0.005, 0.04, 0.001)
sample_sizes = []

for m in mde_range:
    r = power_analysis(baseline_rate, m, alpha, power)
    sample_sizes.append(r['sample_size_per_group'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mde_range * 100, sample_sizes, 'b-', linewidth=2)
ax.set_xlabel('Minimum detectable effect (percentage points)')
ax.set_ylabel('Required sample size per group')
ax.set_title(f'Sample size vs MDE (baseline={baseline_rate:.1%}, power={power:.0%})')
ax.grid(True, alpha=0.3)

# Mark our chosen MDE
our_n = power_analysis(baseline_rate, mde, alpha, power)['sample_size_per_group']
ax.axvline(mde * 100, color='red', linestyle='--', alpha=0.7)
ax.annotate(f'MDE={mde:.0%}, n={our_n:,}',
            xy=(mde * 100, our_n), xytext=(mde * 100 + 0.5, our_n * 1.3),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10)

plt.tight_layout()
plt.show()

## 3. MDE determination

The MDE should be the smallest effect that is practically meaningful.
We approach this from a business perspective.

In [ ]:
# Business-driven MDE calculation
avg_revenue_per_conversion = 45.0  # average order value
monthly_traffic = 100000           # monthly unique visitors

print('MDE determination from business requirements:')
print(f'  Current conversion rate: {baseline_rate:.1%}')
print(f'  Monthly traffic: {monthly_traffic:,}')
print(f'  Avg revenue/conversion: ${avg_revenue_per_conversion:.0f}')
print()

# Calculate revenue impact for different MDE levels
mde_levels = [0.002, 0.005, 0.01, 0.015, 0.02]
print(f'{"MDE":>6s}  {"Rel. MDE":>8s}  {"Monthly rev lift":>16s}  {"Annual rev lift":>15s}  {"n/group":>8s}')
print('-' * 65)
for m in mde_levels:
    rel_mde = m / baseline_rate * 100
    monthly_lift = monthly_traffic * m * avg_revenue_per_conversion
    annual_lift = monthly_lift * 12
    r = power_analysis(baseline_rate, m, alpha, power)
    n = r['sample_size_per_group']
    print(f'{m:.1%}  {rel_mde:7.0f}%  ${monthly_lift:14,.0f}  ${annual_lift:13,.0f}  {n:>8,}')

## 4. Test duration estimation

Given the required sample size and daily traffic, we can estimate
how long the experiment needs to run. We also account for day-of-week
effects by requiring at least one full week.

In [ ]:
daily_traffic = monthly_traffic / 30
traffic_per_variant = daily_traffic / 2  # 50/50 split

required_n = power_analysis(baseline_rate, mde, alpha, power)['sample_size_per_group']
days_needed = np.ceil(required_n / traffic_per_variant)

# Round up to full weeks
weeks_needed = max(1, np.ceil(days_needed / 7))
final_days = int(weeks_needed * 7)

print(f'Test duration estimation:')
print(f'  Daily traffic:          {daily_traffic:,.0f}')
print(f'  Per-variant daily:      {traffic_per_variant:,.0f}')
print(f'  Required n per group:   {required_n:,}')
print(f'  Raw days needed:        {days_needed:.0f}')
print(f'  Rounded to full weeks:  {final_days} days ({int(weeks_needed)} weeks)')
print(f'  Expected total sample:  {int(final_days * daily_traffic):,}')
print()
print('Note: always run for at least 7 full days to capture day-of-week')
print('variation. Avoid stopping early based on peeking at results.')

## 5. Power across different significance levels

Visualizing the power curve helps communicate the tradeoff between
Type I error (alpha) and Type II error (1 - power).

In [ ]:
n_fixed = required_n
effect_range = np.arange(0.002, 0.035, 0.001)

fig, ax = plt.subplots(figsize=(10, 5))

for a in [0.01, 0.05, 0.10]:
    powers = []
    for eff in effect_range:
        p1 = baseline_rate
        p2 = baseline_rate + eff
        p_avg = (p1 + p2) / 2
        se = np.sqrt(p1*(1-p1)/n_fixed + p2*(1-p2)/n_fixed)
        se_h0 = np.sqrt(2 * p_avg * (1 - p_avg) / n_fixed)
        z_crit = stats.norm.ppf(1 - a/2)
        z_effect = eff / se_h0
        pwr = 1 - stats.norm.cdf(z_crit - z_effect) + stats.norm.cdf(-z_crit - z_effect)
        powers.append(pwr)
    ax.plot(effect_range * 100, powers, linewidth=2, label=f'alpha={a}')

ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('True effect size (percentage points)')
ax.set_ylabel('Power')
ax.set_title(f'Power curves at n={n_fixed:,} per group')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Randomization strategy

Proper randomization ensures that the control and treatment groups
are comparable on all observed and unobserved factors.

In [ ]:
# Simulate a randomization check (SRM - Sample Ratio Mismatch)
np.random.seed(42)
n_total = 10000
assignments = np.random.choice(['control', 'treatment'], n_total, p=[0.5, 0.5])

n_control = (assignments == 'control').sum()
n_treatment = (assignments == 'treatment').sum()

# Chi-squared test for SRM
chi2_stat = (n_control - n_total/2)**2 / (n_total/2) + \
            (n_treatment - n_total/2)**2 / (n_total/2)
srm_p_value = 1 - stats.chi2.cdf(chi2_stat, df=1)

print('Randomization integrity check (SRM test):')
print(f'  Control:   {n_control:,} ({n_control/n_total:.1%})')
print(f'  Treatment: {n_treatment:,} ({n_treatment/n_total:.1%})')
print(f'  Chi-squared: {chi2_stat:.3f}')
print(f'  p-value:     {srm_p_value:.4f}')
print(f'  SRM detected: {"Yes" if srm_p_value < 0.01 else "No"}')
print()
print('Best practices for randomization:')
print('  1. Hash-based assignment using user_id ensures deterministic grouping')
print('  2. Run SRM check daily to detect assignment bugs early')
print('  3. Stratify by key segments (device, geography) if needed')
print('  4. Exclude bots and internal traffic before randomization')

## 7. Multiple testing correction

When running multiple experiments or testing multiple metrics,
the probability of at least one false positive increases.
We discuss three correction methods.

In [ ]:
# Simulate p-values from 5 metrics tested in one experiment
p_values = [0.003, 0.042, 0.15, 0.51, 0.87]
metric_names = ['conversion', 'revenue', 'session_duration', 'pages_viewed', 'bounce_rate']

print('Multiple testing correction comparison:')
print(f'{"Metric":20s}  {"Raw p":>7s}  {"Bonferroni":>10s}  {"Holm":>7s}  {"BH-FDR":>7s}')
print('-' * 60)

bonf = multiple_comparison_correction(p_values, 'bonferroni')
holm = multiple_comparison_correction(p_values, 'holm')
bh = multiple_comparison_correction(p_values, 'bh_fdr')

for i, name in enumerate(metric_names):
    raw = p_values[i]
    b = bonf['adjusted_p_values'][i]
    h = holm['adjusted_p_values'][i]
    fdr = bh['adjusted_p_values'][i]
    print(f'{name:20s}  {raw:7.4f}  {b:10.4f}  {h:7.4f}  {fdr:7.4f}')

print(f'\nRejections at alpha=0.05:')
print(f'  No correction:  {sum(p < 0.05 for p in p_values)}')
print(f'  Bonferroni:     {bonf["n_rejected"]}')
print(f'  Holm:           {holm["n_rejected"]}')
print(f'  BH-FDR:         {bh["n_rejected"]}')

In [ ]:
# Family-wise error rate illustration
n_tests = np.arange(1, 21)
fwer = 1 - (1 - 0.05) ** n_tests

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_tests, fwer * 100, 'b-o', markersize=5, linewidth=2)
ax.axhline(5, color='red', linestyle='--', alpha=0.5, label='Nominal alpha (5%)')
ax.set_xlabel('Number of independent tests')
ax.set_ylabel('Family-wise error rate (%)')
ax.set_title('False positive inflation without correction')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 70)

ax.annotate(f'{fwer[9]:.0%} at 10 tests',
            xy=(10, fwer[9]*100), xytext=(13, fwer[9]*100 + 8),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10)

plt.tight_layout()
plt.show()

print(f'With 10 independent tests at alpha=0.05, the probability of')
print(f'at least one false positive is {fwer[9]:.0%}.')

## 8. Design checklist

Before launching any experiment, verify:

1. **Hypothesis**: clearly stated, with a primary metric identified
2. **Sample size**: calculated via power analysis with practical MDE
3. **Duration**: at least 1-2 full weeks to capture weekly cycles
4. **Randomization**: hash-based, with SRM monitoring
5. **Guardrail metrics**: revenue, latency, error rates monitored but
   not used for stopping decisions
6. **Analysis plan**: pre-registered to prevent p-hacking
7. **Multiple testing**: correction method selected before launch

In [ ]:
# Summary table for our three experiments
experiments = pd.DataFrame({
    'Experiment': ['exp_001 (Website redesign)', 'exp_002 (Pricing change)', 'exp_003 (Email subject)'],
    'Baseline rate': ['3.0%', '2.1%', '12.0%'],
    'Expected effect': ['+1.2pp', '+0.2pp', '+3.0pp'],
    'MDE (planned)': ['1.0pp', '0.5pp', '2.0pp'],
    'N per group': [5000, 5000, 5000],
    'Power at MDE': ['~80%', '<50%', '>90%'],
})

print('Experiment design summary:')
experiments.set_index('Experiment')

## Summary

Proper experiment design is the foundation of trustworthy A/B testing.
Key takeaways:

- **Power analysis** shows that detecting a 1pp lift on a 3% baseline
  requires ~4,000-5,000 users per group at 80% power
- **MDE** should be set based on the minimum business-meaningful effect,
  not the expected effect
- **Test duration** must account for weekly cycles and avoid early stopping
- **Multiple testing** inflates false positive rates and requires correction
  when testing multiple metrics or running concurrent experiments

Next: run frequentist, Bayesian, and sequential tests in `03_statistical_tests.ipynb`.